In [1]:
# @title Setup (chạy ô này trước)
# Colab bắt đầu với một máy trống — clone repo và cài dependency.
import os, subprocess, sys

REPO = "https://github.com/hieutrungdao/Day21-Track3-Finetuning-Lab.git"
if not os.path.exists("Day21-Track3-Finetuning-Lab"):
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir("Day21-Track3-Finetuning-Lab")
sys.path.insert(0, "src")

# Install from requirements.txt, NOT a copied list. The copied list is how the
# torchao>=0.16 pin reached requirements.txt and this bootstrap on different days --
# and a bootstrap missing a pin does not fail here, it fails 10 minutes later inside
# get_peft_model(). One source of truth. torch is preinstalled on Colab and
# requirements.txt pins it compatibly, so that line is a no-op.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

os.environ.setdefault("COMPUTE_TIER", "T4")
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE — Runtime > Change runtime type > T4 GPU")


GPU: Tesla T4


# NB2 — Đóng băng eval & đo BA baseline (trước khi train)

> Deck §17: *điểm không nằm ở việc perplexity giảm bao nhiêu, mà ở việc bạn có chứng
> minh được bản fine-tune thắng baseline (b) hay không.*

**Thứ tự quan trọng.** Đo baseline **trước** khi train, không phải sau. Nếu đo sau,
bạn sẽ (một cách vô thức) chỉnh prompt baseline cho tới khi fine-tune của mình thắng.
Đó là lý do notebook này chạy trước NB3.

Ba baseline:
| | Là gì | Vì sao có mặt |
|---|---|---|
| **(a)** | base + prompt ngây thơ | mốc sàn |
| **(b)** | base + prompt **đã tối ưu** | **mốc thật sự phải vượt** |
| (c) | bản fine-tune | đo ở NB5 |

In [2]:
import json, os, pathlib, sys
sys.path.insert(0, str(pathlib.Path.cwd() / "src"))
sys.path.insert(0, str(pathlib.Path.cwd().parent / "src"))

from labkit import evaluate as ev, generate, report
from labkit.config import get_tier

ROOT = pathlib.Path.cwd() if (pathlib.Path.cwd() / "data").exists() else pathlib.Path.cwd().parent
TIER = get_tier(os.environ.get("COMPUTE_TIER", "T4"))

def load_jsonl(p):
    return [json.loads(l) for l in open(p, encoding="utf-8") if l.strip()]

target = load_jsonl(ROOT / "data" / "eval_target.jsonl")
regression = load_jsonl(ROOT / "data" / "eval_regression.jsonl")

# EVAL_LIMIT truncates BOTH eval sets — a smoke mode for slow hardware. It is recorded
# in results/ so the grader can see the run was abbreviated; a submitted run must use
# the full sets (leave EVAL_LIMIT unset).
EVAL_LIMIT = int(os.environ.get("EVAL_LIMIT", "0"))
if EVAL_LIMIT:
    target, regression = target[:EVAL_LIMIT], regression[:EVAL_LIMIT]
    print(f"⚠ EVAL_LIMIT={EVAL_LIMIT} — SMOKE MODE, not a submittable run")
print(f"target={len(target)}  regression={len(regression)}  tier={TIER.name}")

target=50  regression=15  tier=T4


## 1. Nạp base model (chưa fine-tune)

In [3]:
model, tok = generate.load_base(TIER)
generate.free_memory()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/2.76k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/15.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/5.23M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/3.35M [00:00<?, ?B/s]

...

chat_template.jinja:   0%|          | 0.00/7.99k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

## 2. Chấm một baseline trên cả bốn nhóm

Bốn nhóm: **target · regression · format · latency**. Cùng một hàm cho mọi run —
baseline hay fine-tune — nên các con số so sánh được với nhau.

In [4]:
def score_run(model, tok, system_prompt, label):
    prompts = [r["input"] for r in target]
    preds, lat = generate.generate_batch(model, tok, prompts, system=system_prompt,
                                         label=f"{label}/target")

    tgt = sum(ev.triage_field_accuracy(p, r["label"]) for p, r in zip(preds, target)) / len(target)
    fmt = sum(ev.has_required_keys(p, ev.TRIAGE_KEYS) for p in preds) / len(preds)

    rprompts = [r["instruction"] for r in regression]
    rpreds, _ = generate.generate_batch(model, tok, rprompts, system=None, max_new_tokens=96,
                                        label=f"{label}/regression")
    reg = sum(ev.keyword_recall(p, r["keywords"]) for p, r in zip(rpreds, regression)) / len(regression)

    scores = ev.GroupScores(target=tgt, regression=reg, format=fmt, latency_ms=lat, n=len(target))
    print(f"{label:28s} target={tgt:.3f}  regression={reg:.3f}  format={fmt:.3f}  {lat:.0f}ms")
    return scores, preds, rpreds


scores_a, preds_a, _ = score_run(model, tok, generate.NAIVE_PROMPT, "(a) base + naive prompt")
scores_b, preds_b, rpreds_b = score_run(model, tok, generate.OPTIMIZED_PROMPT, "(b) base + optimized prompt")

  [(a) base + naive prompt/target] batch 1/13     16s elapsed  ~  190s left
  [(a) base + naive prompt/target] batch 2/13     29s elapsed  ~  157s left
  [(a) base + naive prompt/target] batch 3/13     41s elapsed  ~  136s left
  [(a) base + naive prompt/target] batch 4/13     54s elapsed  ~  121s left
  [(a) base + naive prompt/target] batch 5/13     67s elapsed  ~  106s left
  [(a) base + naive prompt/target] batch 6/13     79s elapsed  ~   93s left
  [(a) base + naive prompt/target] batch 7/13     92s elapsed  ~   79s left
  [(a) base + naive prompt/target] batch 8/13    105s elapsed  ~   66s left
  [(a) base + naive prompt/target] batch 9/13    118s elapsed  ~   52s left
  [(a) base + naive prompt/target] batch 10/13    130s elapsed  ~   39s left
  [(a) base + naive prompt/target] batch 11/13    143s elapsed  ~   26s left
  [(a) base + naive prompt/target] batch 12/13    156s elapsed  ~   13s left
  [(a) base + naive prompt/target] batch 13/13    169s elapsed  ~    0s left
  [(a) b

## 3. Đóng băng

Từ đây **không được sửa** `eval_target.jsonl`, `eval_regression.jsonl`, hay
`OPTIMIZED_PROMPT` nữa. Sửa bất kỳ thứ nào sau khi biết kết quả fine-tune = tự lừa mình.

In [5]:
frozen = {
    "tier": TIER.name,
    "model": TIER.model_id,
    "baseline_a": scores_a.as_dict(),
    "baseline_b": scores_b.as_dict(),
    "optimized_prompt_sha": __import__("hashlib").sha256(
        generate.OPTIMIZED_PROMPT.encode()).hexdigest()[:16],
    "n_target": len(target),
    "n_regression": len(regression),
    "eval_limit": EVAL_LIMIT or None,
    "smoke_mode": bool(EVAL_LIMIT),
}
report.write_json(frozen, "baselines_frozen.json", results_dir=ROOT / "results")
print(json.dumps(frozen, ensure_ascii=False, indent=2))

{
  "tier": "T4",
  "model": "unsloth/Qwen3.5-4B",
  "baseline_a": {
    "target": 0.0,
    "regression": 0.7577777777777778,
    "format": 0.0,
    "latency_ms": 3371.772553240005,
    "n": 50,
    "extra": {}
  },
  "baseline_b": {
    "target": 0.765,
    "regression": 0.7577777777777778,
    "format": 1.0,
    "latency_ms": 1050.1945459399894,
    "n": 50,
    "extra": {}
  },
  "optimized_prompt_sha": "719e74d3b6232053",
  "n_target": 50,
  "n_regression": 15,
  "eval_limit": null,
  "smoke_mode": false
}


### Đọc kết quả trước khi đi tiếp

* **(b) đã cao sẵn?** Tốt — bài toán của bạn có thể *không cần* fine-tune. Đó là một
  kết luận hợp lệ và được chấm điểm đầy đủ (deck §1).
* **(b) ≈ (a)?** Prompt "tối ưu" của bạn chưa đủ tốt. Cải thiện nó **bây giờ**, trước
  khi train — nếu không, phần thắng ở NB5 sẽ là ảo.


## ✅ Checkpoint NB2
- [ ] `results/baselines_frozen.json` có cả (a) và (b)
- [ ] Bạn đã đọc và chấp nhận con số (b) — **trước** khi thấy bất kỳ kết quả train nào